# Phần 0: Chuẩn bị Môi trường

Trong bài lab này, chúng ta sẽ xây dựng một hệ thống RAG (Retrieval-Augmented Generation) từ các thành phần cơ bản.  
Để bắt đầu, chúng ta cần chuẩn bị môi trường làm việc.

Quá trình này gồm 2 bước chính:

1. **Khởi chạy Dịch vụ (với Docker):**  
   Chúng ta cần chạy 2 cơ sở dữ liệu là Elasticsearch và Weaviate.  
   Cách dễ nhất để chạy chúng trên máy local là sử dụng Docker.

2. **Cài đặt Thư viện (với Python):**  
   Cài đặt các thư viện Python để tương tác với dữ liệu, chunking, và các dịch vụ đã cài.

---

## 0.1. Khởi chạy Dịch vụ với Docker

Chúng ta sẽ sử dụng `docker-compose` để khởi chạy đồng thời 2 dịch vụ:  
Elasticsearch (cho text search) và Weaviate (một vector DB mà chúng ta sẽ dùng tính năng text search).

### Bước 1: Tạo file `docker-compose.yml`

```yaml
version: '3.9'

services:
  # Dịch vụ Elasticsearch
  elasticsearch:
    image: elasticsearch:8.11.0 # Sử dụng một phiên bản cụ thể
    container_name: es-lab
    environment:
      # Chạy ở chế độ single-node
      - discovery.type=single-node
      # Tắt security cho mục đích lab (KHÔNG làm điều này trong production)
      - xpack.security.enabled=false
      # Cấp phát 1GB RAM cho Elasticsearch
      - "ES_JAVA_OPTS=-Xms1g -Xmx1g"
    ports:
      - "9200:9200"   # Port REST API
      - "9300:9300"   # Port giao tiếp nội bộ
    volumes:
      - es_data:/usr/share/elasticsearch/data
    networks:
      - rag_lab_net

  # Dịch vụ Weaviate
  weaviate:
    image: semitechnologies/weaviate:1.24.1 # Sử dụng một phiên bản cụ thể
    container_name: weaviate-lab
    ports:
      - "8080:8080"   # Port REST API
      - "50051:50051" # Port gRPC
    volumes:
      - weaviate_data:/var/lib/weaviate
    environment:
      # Cho phép truy cập không cần xác thực
      AUTHENTICATION_ANONYMOUS_ENABLED: 'true'
      # Tắt module vectorizer mặc định (chúng ta sẽ tự quản lý)
      DEFAULT_VECTORIZER_MODULE: 'none'
      PERSISTENCE_DATA_PATH: '/var/lib/weaviate'
      CLUSTER_HOSTNAME: 'localhost'
    networks:
      - rag_lab_net

# Định nghĩa volumes để lưu trữ dữ liệu
volumes:
  es_data:
  weaviate_data:

# Định nghĩa network chung
networks:
  rag_lab_net:
```

### Bước 2: Khởi chạy Docker

Mở terminal tại thư mục chứa file `docker-compose.yml` và chạy lệnh:

```bash
docker compose up -d
```

### Bước 3: Kiểm tra Dịch vụ

Sau khi chạy xong, bạn có thể kiểm tra xem các dịch vụ đã hoạt động chưa:

- Elasticsearch: Mở trình duyệt và truy cập http://localhost:9200. Bạn sẽ thấy một file JSON trả về thông tin của cluster (ví dụ: "name" : "es-lab", "version" : { "number" : "8.11.0", ... }).

- Weaviate: Mở trình duyệt và truy cập http://localhost:8080/v1

Nếu thấy các kết quả trên, bạn đã sẵn sàng phần dịch vụ!


In [ ]:
!pip install -r requirements.txt

## 0.2. Kiểm tra Kết nối

In [5]:
import sys
import elasticsearch
import weaviate
import requests
from duckduckgo_search import DDGS

print(f"Phiên bản Python: {sys.version.split()[0]}")

# --- 1. Kiểm tra kết nối Elasticsearch ---
try:
    es_client = elasticsearch.Elasticsearch("http://localhost:9200")
    if es_client.ping():
        print("✅ Kết nối Elasticsearch thành công!")
        es_info = es_client.info()
        print(f"   Phiên bản ES: {es_info['version']['number']}")
    else:
        print("❌ Lỗi: Không thể ping Elasticsearch.")
except Exception as e:
    print(f"❌ Lỗi khi kết nối Elasticsearch:\n{e}")
    print("   Vui lòng kiểm tra lại dịch vụ 'es-lab' trong Docker.")


# --- 2. Kiểm tra kết nối Weaviate ---
try:
    # Sử dụng client v4
    weaviate_client = weaviate.connect_to_local(
        host="localhost",
        port=8080,
        grpc_port=50051
    )
    if weaviate_client.is_ready():
        print("\n✅ Kết nối Weaviate thành công!")
        meta = weaviate_client.get_meta()
        print(f"   Phiên bản Weaviate: {meta['version']}")
    else:
        print("❌ Lỗi: Weaviate báo chưa sẵn sàng.")
except Exception as e:
    print(f"❌ Lỗi khi kết nối Weaviate:\n{e}")
    print("   Vui lòng kiểm tra lại dịch vụ 'weaviate-lab' trong Docker.")
finally:
    # Đóng kết nối client v4
    if 'weaviate_client' in locals() and weaviate_client.is_connected():
        weaviate_client.close()


# --- 3. Kiểm tra các thư viện khác ---
try:
    with DDGS() as ddgs:
        print("\n✅ Thư viện DuckDuckGo Search sẵn sàng.")
    
    import langchain
    import rank_bm25
    import wikipediaapi
    print("✅ Các thư viện LangChain, BM25, Wikipedia đã được import thành công.")
except ImportError as e:
    print(f"❌ Lỗi import thư viện: {e}")

print("\n--- Môi trường đã sẵn sàng cho bài Lab! ---")

Phiên bản Python: 3.12.9
✅ Kết nối Elasticsearch thành công!
   Phiên bản ES: 8.11.0

✅ Kết nối Weaviate thành công!
   Phiên bản Weaviate: 1.34.0

✅ Thư viện DuckDuckGo Search sẵn sàng.
✅ Các thư viện LangChain, BM25, Wikipedia đã được import thành công.

--- Môi trường đã sẵn sàng cho bài Lab! ---


C:\Users\Admin\AppData\Local\Temp\ipykernel_15272\1658461538.py:48: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


# Phần 1: Tải và Tiền xử lý Dữ liệu

In [6]:
import wikipediaapi
import time

def load_wiki_pages(page_titles):
    """
    Tải nội dung các trang Wikipedia tiếng Việt từ danh sách page_titles.
    """
    # 1. Khởi tạo Wikipedia API cho tiếng Việt (vi)
    # Thêm user_agent để tuân thủ chính sách của Wikipedia
    wiki = wikipediaapi.Wikipedia(
        language='vi',
        extract_format=wikipediaapi.ExtractFormat.WIKI,
        user_agent="GenAI_Lab/1.0 (example@email.com)" 
        # Bạn có thể thay đổi user_agent nếu muốn
    )
    
    documents = []
    print(f"Bắt đầu tải {len(page_titles)} trang...")

    # 2. Lặp qua từng tiêu đề trang và tải dữ liệu
    for title in page_titles:
        page = wiki.page(title)
        
        if not page.exists():
            print(f"!!! Cảnh báo: Trang '{title}' không tồn tại.")
            continue
            
        # 3. Lưu trữ dữ liệu
        # Chúng ta sẽ lưu 3 thông tin quan trọng
        doc = {
            "title": page.title,
            "text": page.text, # Đây là nội dung text đầy đủ của trang
            "url": page.fullurl
        }
        documents.append(doc)
        
        print(f"✅ Đã tải xong: {page.title} (Kích thước: {len(page.text)} ký tự)")
        
        # Thêm một chút thời gian nghỉ để không spam API
        time.sleep(0.5) 
        
    return documents

# --- Đây là danh sách các trang chúng ta sẽ sử dụng ---
# Cảm thấy thoải mái thêm hoặc bớt các trang liên quan
PAGE_TITLES = [
    "Blockchain",
    "Tiền mã hóa",
    "Bitcoin",
    "Ethereum",
    "Hợp đồng thông minh",
    "Web3"
]

# Chạy hàm để tải dữ liệu
documents = load_wiki_pages(PAGE_TITLES)

print(f"\n--- Tải hoàn tất ---")
print(f"Tổng cộng đã tải về {len(documents)} tài liệu.")

Bắt đầu tải 6 trang...
✅ Đã tải xong: Blockchain (Kích thước: 6284 ký tự)
✅ Đã tải xong: Tiền mã hóa (Kích thước: 3680 ký tự)
✅ Đã tải xong: Bitcoin (Kích thước: 45075 ký tự)
✅ Đã tải xong: Ethereum (Kích thước: 14633 ký tự)
!!! Cảnh báo: Trang 'Hợp đồng thông minh' không tồn tại.
!!! Cảnh báo: Trang 'Web3' không tồn tại.

--- Tải hoàn tất ---
Tổng cộng đã tải về 4 tài liệu.


Sau khi chạy cell trên, biến documents của chúng ta là một danh sách các dictionary. Hãy kiểm tra xem dữ liệu trông như thế nào.

In [7]:
# Kiểm tra tài liệu đầu tiên (Blockchain)
if documents:
    print(f"--- Thông tin tài liệu đầu tiên ---")
    doc_0 = documents[0]
    print(f"Tiêu đề (title): {doc_0['title']}")
    print(f"URL: {doc_0['url']}")
    
    # In ra 500 ký tự đầu tiên của nội dung
    print(f"\nNội dung (text) - 500 ký tự đầu:")
    print(doc_0['text'][:500] + "...")
else:
    print("Không có tài liệu nào được tải về. Vui lòng kiểm tra lại code hoặc kết nối mạng.")

--- Thông tin tài liệu đầu tiên ---
Tiêu đề (title): Blockchain
URL: https://vi.wikipedia.org/wiki/Blockchain

Nội dung (text) - 500 ký tự đầu:
Blockchain (chuỗi khối), tên ban đầu block chain là một cơ sở dữ liệu phân cấp lưu trữ thông tin trong các khối thông tin được liên kết với nhau bằng mã hóa và mở rộng theo thời gian. Mỗi khối thông tin đều chứa thông tin về thời gian khởi tạo và được liên kết tới khối trước đó, kèm một mã thời gian và dữ liệu giao dịch. Blockchain được thiết kế để chống lại sự thay đổi của dữ liệu: Một khi dữ liệu đã được mạng lưới chấp nhận thì sẽ không có cách nào thay đổi được nó.

Tổng quan
Blockchain được ...


# Phần 2: So sánh các Chiến lược Chunking

## 2.1 Chuẩn bị dữ liêu chunking

In [8]:
from langchain_core.documents import Document

# Lấy dữ liệu 'documents' từ Phần 1
# documents = [
#   {"title": "Blockchain", "text": "...", "url": "..."},
#   ...
# ]

# Tìm tài liệu "Blockchain"
doc_blockchain_raw = None
for doc in documents:
    if doc['title'].lower() == "blockchain":
        doc_blockchain_raw = doc
        break

if doc_blockchain_raw:
    # Đây là đối tượng Document LangChain gốc
    doc_blockchain_langchain = Document(
        page_content=doc_blockchain_raw['text'],
        metadata={"title": doc_blockchain_raw['title'], "url": doc_blockchain_raw['url']}
    )
    print(f"Đã chọn tài liệu '{doc_blockchain_raw['title']}' để so sánh.")
    print(f"Tổng số ký tự: {len(doc_blockchain_raw['text'])} ký tự.")
else:
    print("Lỗi: Không tìm thấy tài liệu 'Blockchain'. Vui lòng chạy lại Phần 1.")

# Hàm trợ giúp để in kết quả chunking cho đẹp
def print_chunks(chunks, strategy_name, char_limit=250):
    print(f"\n--- Kết quả từ [ {strategy_name} ] ---")
    print(f"Tổng số chunks: {len(chunks)}")
    
    for i, chunk in enumerate(chunks[:2]): # Chỉ in 2 chunk đầu tiên
        print(f"\n[Chunk {i+1}/{len(chunks)}]")
        
        # Một số splitter (như Markdown) sẽ thêm metadata
        if 'metadata' in chunk:
            print(f"Metadata (mới): {chunk.metadata}")
        else:
            print(f"Metadata (kế thừa): {chunk.metadata}")
            
        print(f"Kích thước chunk: {len(chunk.page_content)} ký tự")
        print("Nội dung (trích đoạn):")
        print(chunk.page_content[:char_limit] + "...")

Đã chọn tài liệu 'Blockchain' để so sánh.
Tổng số ký tự: 6284 ký tự.


### 2.3.1. Chiến lược 1: CharacterTextSplitter

In [9]:
from langchain_text_splitters import CharacterTextSplitter

# 1. Khởi tạo
char_splitter = CharacterTextSplitter(
    separator="\n\n", # Chỉ chia khi gặp 2 dấu xuống dòng
    chunk_size=1000,   # Kích thước chunk mong muốn
    chunk_overlap=100,
    length_function=len
)

# 2. Chạy
# Lưu ý: .split_documents() yêu cầu một list, nên ta bọc nó lại
chunks_char = char_splitter.split_documents([doc_blockchain_langchain])

# 3. In kết quả
print_chunks(chunks_char, "CharacterTextSplitter")

d:\anaconda3\envs\falcon\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Created a chunk of size 1062, which is longer than the specified 1000



--- Kết quả từ [ CharacterTextSplitter ] ---
Tổng số chunks: 9

[Chunk 1/9]
Metadata (kế thừa): {'title': 'Blockchain', 'url': 'https://vi.wikipedia.org/wiki/Blockchain'}
Kích thước chunk: 472 ký tự
Nội dung (trích đoạn):
Blockchain (chuỗi khối), tên ban đầu block chain là một cơ sở dữ liệu phân cấp lưu trữ thông tin trong các khối thông tin được liên kết với nhau bằng mã hóa và mở rộng theo thời gian. Mỗi khối thông tin đều chứa thông tin về thời gian khởi tạo và đượ...

[Chunk 2/9]
Metadata (kế thừa): {'title': 'Blockchain', 'url': 'https://vi.wikipedia.org/wiki/Blockchain'}
Kích thước chunk: 1062 ký tự
Nội dung (trích đoạn):
Tổng quan
Blockchain được đảm bảo nhờ cách thiết kế sử dụng hệ thống tính toán phân cấp với khả năng chịu lỗi byzantine cao. Nhờ thế nên Blockchain có thể đạt được sự đồng thuận phân cấp. Vì vậy Blockchain phù hợp để ghi lại những sự kiện, hồ sơ y tế...


## 2.3.2 Chiến lược 2: RecursiveCharacterTextSplitter

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Khởi tạo
recursive_splitter = RecursiveCharacterTextSplitter(
    # Danh sách separators được thử lần lượt
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

# 2. Chạy
chunks_recursive = recursive_splitter.split_documents([doc_blockchain_langchain])

# 3. In kết quả
print_chunks(chunks_recursive, "RecursiveCharacterTextSplitter")


--- Kết quả từ [ RecursiveCharacterTextSplitter ] ---
Tổng số chunks: 10

[Chunk 1/10]
Metadata (kế thừa): {'title': 'Blockchain', 'url': 'https://vi.wikipedia.org/wiki/Blockchain'}
Kích thước chunk: 472 ký tự
Nội dung (trích đoạn):
Blockchain (chuỗi khối), tên ban đầu block chain là một cơ sở dữ liệu phân cấp lưu trữ thông tin trong các khối thông tin được liên kết với nhau bằng mã hóa và mở rộng theo thời gian. Mỗi khối thông tin đều chứa thông tin về thời gian khởi tạo và đượ...

[Chunk 2/10]
Metadata (kế thừa): {'title': 'Blockchain', 'url': 'https://vi.wikipedia.org/wiki/Blockchain'}
Kích thước chunk: 425 ký tự
Nội dung (trích đoạn):
Tổng quan
Blockchain được đảm bảo nhờ cách thiết kế sử dụng hệ thống tính toán phân cấp với khả năng chịu lỗi byzantine cao. Nhờ thế nên Blockchain có thể đạt được sự đồng thuận phân cấp. Vì vậy Blockchain phù hợp để ghi lại những sự kiện, hồ sơ y tế...


Đây là splitter "tiêu chuẩn vàng" và được khuyến nghị sử dụng trong hầu hết các trường hợp.

Cách hoạt động: Thay vì 1 separator, nó nhận một DANH SÁCH các separator, theo thứ tự ưu tiên (ví dụ: ["\n\n", "\n", ". ", " ", ""]).

Logic:

- Nó thử chia văn bản bằng "\n\n" (đoạn văn).

- Nếu một chunk (đoạn văn) vẫn còn quá dài so với chunk_size, nó sẽ lấy chunk đó và thử chia tiếp bằng separator tiếp theo ("\n" - dòng).

- Nếu vẫn quá dài, nó thử chia bằng ". " (câu).

- Cứ như vậy cho đến khi chunk nhỏ hơn chunk_size.

Ưu điểm: Rất linh hoạt, luôn cố gắng giữ ngữ nghĩa (đoạn > dòng > câu) và đảm bảo các chunk không vượt quá chunk_size.

## 2.3.3. Chiến lược 3: MarkdownHeaderTextSplitter

In [11]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 1. Định nghĩa cấu trúc Header
# Văn bản Wiki dùng == và ===, tương ứng với # và ## trong Markdown
headers_to_split_on = [
    ("==", "Header 1"), # Đặt tên "Header 1" cho tiêu đề cấp ==
    ("===", "Header 2"), # Đặt tên "Header 2" cho tiêu đề cấp ===
]

# 2. Khởi tạo
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    return_each_line=False # Gộp các dòng lại
)

# 3. Chạy
# Quan trọng: Splitter này chạy trên text GỐC, không phải Document
# Nó sẽ TỰ TẠO ra các Document mới với metadata
chunks_markdown = markdown_splitter.split_text(doc_blockchain_raw['text'])

# 4. In kết quả (Hàm print_chunks() của chúng ta cũng hoạt động)
# Chúng ta sẽ thấy metadata được tự động điền
print_chunks(chunks_markdown, "MarkdownHeaderTextSplitter")

# In thêm 1 chunk ở giữa để thấy metadata
print("\n... (Kiểm tra một chunk ở giữa) ...")
if len(chunks_markdown) > 5:
    chunk_5 = chunks_markdown[5]
    print(f"\n[Chunk 6/{len(chunks_markdown)}]")
    print(f"Metadata (mới): {chunk_5.metadata}")
    print(f"Kích thước chunk: {len(chunk_5.page_content)} ký tự")
    print("Nội dung (trích đoạn):")
    print(chunk_5.page_content[:250] + "...")


--- Kết quả từ [ MarkdownHeaderTextSplitter ] ---
Tổng số chunks: 1

[Chunk 1/1]
Metadata (kế thừa): {}
Kích thước chunk: 6274 ký tự
Nội dung (trích đoạn):
Blockchain (chuỗi khối), tên ban đầu block chain là một cơ sở dữ liệu phân cấp lưu trữ thông tin trong các khối thông tin được liên kết với nhau bằng mã hóa và mở rộng theo thời gian. Mỗi khối thông tin đều chứa thông tin về thời gian khởi tạo và đượ...

... (Kiểm tra một chunk ở giữa) ...


## Tóm lai sử dụng RecursiveCharacterTextSplitter

In [12]:
# Chúng ta sẽ đi tiếp với bộ chunks từ Recursive splitter
# Nhưng chúng ta cần chạy nó trên TẤT CẢ các tài liệu

print("Đang thực hiện chunking TẤT CẢ tài liệu (sử dụng Recursive)...")

# Chuyển đổi tất cả docs sang định dạng LangChain
all_langchain_docs = [
    Document(
        page_content=doc['text'], 
        metadata={"title": doc['title'], "url": doc['url']}
    ) 
    for doc in documents
]

# Sử dụng lại recursive_splitter đã khởi tạo ở trên
final_chunks = recursive_splitter.split_documents(all_langchain_docs)

print(f"Hoàn tất chunking! Đã tạo ra {len(final_chunks)} chunks.")

Đang thực hiện chunking TẤT CẢ tài liệu (sử dụng Recursive)...
Hoàn tất chunking! Đã tạo ra 110 chunks.


# Phần 3: Text Search với BM25 (Sparse Retrieval)

In [13]:
from rank_bm25 import BM25Okapi
import re
from tqdm import tqdm # Thư viện để xem thanh tiến trình (progress bar)

# 1. Chuẩn bị "kho văn bản" (corpus)
# BM25 chỉ cần một danh sách các nội dung text (page_content)
print(f"Chuẩn bị corpus từ {len(final_chunks)} chunks...")
corpus = [chunk.page_content for chunk in final_chunks]

# 2. Tokenizer (Tách từ)
# BM25 hoạt động dựa trên "từ" (token). 
# Chúng ta sẽ dùng một hàm tách từ đơn giản cho tiếng Việt.
# Lưu ý: Trong thực tế, bạn nên dùng thư viện tốt hơn 
# như underthesea hoặc vncorenlp, nhưng hàm này đủ cho bài lab.
def simple_tokenizer(text):
    # Xóa dấu câu, chuyển về chữ thường
    text = re.sub(r'[^\w\s]', '', text.lower()) 
    # Tách từ bằng khoảng trắng
    return text.split()

print("Đang token hóa corpus (có thể mất vài giây)...")
# Chúng ta dùng tqdm để xem tiến trình
tokenized_corpus = [simple_tokenizer(doc) for doc in tqdm(corpus)]

# 3. Khởi tạo và "fit" model BM25
print("\nĐang train (fit) model BM25...")
bm25 = BM25Okapi(tokenized_corpus)
print("Đã train xong!")

Chuẩn bị corpus từ 110 chunks...
Đang token hóa corpus (có thể mất vài giây)...


100%|██████████| 110/110 [00:00<00:00, 14254.44it/s]


Đang train (fit) model BM25...
Đã train xong!


In [14]:
# 1. Đặt câu truy vấn
query = "Hợp đồng thông minh là gì?"
top_k = 3 # Chúng ta muốn lấy 3 kết quả hàng đầu

# 2. Tokenize câu query (PHẢI dùng cùng tokenizer)
tokenized_query = simple_tokenizer(query)
print(f"Query đã token hóa: {tokenized_query}")

# 3. Lấy scores và top indices
# Dùng .get_top_n() để lấy N kết quả tốt nhất
# Nó sẽ trả về danh sách các document (là text)
top_n_results = bm25.get_top_n(tokenized_query, corpus, n=top_k)

# Hoặc, lấy index để chúng ta có thể truy cập metadata
# (Cách này tốt hơn)
scores = bm25.get_scores(tokenized_query)

# Sắp xếp và lấy top_k indices
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

# 4. In kết quả
print(f"\n--- {top_k} kết quả hàng đầu cho query: '{query}' ---")

for i, idx in enumerate(top_indices):
    # Lấy lại chunk gốc (với metadata) từ `final_chunks`
    chunk = final_chunks[idx] 
    score = scores[idx]
    
    print(f"\n[Kết quả {i+1}] (Index: {idx}, Score BM25: {score:.2f})")
    print(f"Nguồn: {chunk.metadata['title']}")
    print(f"URL: {chunk.metadata['url']}")
    print("Nội dung chunk (300 ký tự đầu):")
    print(chunk.page_content[:300].strip() + "...")
    print("-" * 20)

Query đã token hóa: ['hợp', 'đồng', 'thông', 'minh', 'là', 'gì']

--- 3 kết quả hàng đầu cho query: 'Hợp đồng thông minh là gì?' ---

[Kết quả 1] (Index: 101, Score BM25: 7.19)
Nguồn: Ethereum
URL: https://vi.wikipedia.org/wiki/Ethereum
Nội dung chunk (300 ký tự đầu):
Lưu ý rằng "hợp đồng" trong Ethereum không phải là một cái gì đó phải "hoàn thành" hoặc "tuân thủ". Thay vào đó, nó giống như các "thực thể tự trị" sống bên trong môi trường Ethereum, luôn thực hiện một đoạn mã cụ thể khi được tác động bởi một thông điệp hoặc giao dịch, và có quyền kiểm soát trực số...
--------------------

[Kết quả 2] (Index: 6, Score BM25: 3.93)
Nguồn: Blockchain
URL: https://vi.wikipedia.org/wiki/Blockchain
Nội dung chunk (300 ký tự đầu):
Hợp đồng thông minh (smart contracts) và tài sản thông minh
Hợp đồng thông minh là các khối để xây dựng nên các ứng dụng phi tập trung. Một hợp đồng thông minh tương đương với một chương trình nhỏ mà bạn có thể tin tưởng với một đơn vị giá trị và quản lý giá trị đó. Ý

In [15]:
# [BÀI TẬP]

# 1. Đặt câu truy vấn
student_query = "Bitcoin và Web3 khác nhau như thế nào?"
top_k_student = 2

# [CODE CỦA HỌC VIÊN TẠI ĐÂY]
# Gợi ý:
# 1. Sử dụng hàm 'simple_tokenizer' để tokenize câu 'student_query'.
# 2. Dùng 'bm25.get_scores(...)' để lấy điểm cho query mới.
# 3. Sắp xếp scores và lấy 'top_k_student' indices hàng đầu (giống ví dụ trên).
# 4. Lặp qua các indices, lấy chunk từ 'final_chunks'
# 5. In kết quả (metadata và nội dung)

# (Học viên sẽ code ở đây)

# Phần 4: Index và Search với Weaviate (Sử dụng Native Client)

1. Định nghĩa Schema: Tạo một "Collection" (giống như một table/index) trong Weaviate để lưu trữ các chunk của chúng ta với nhiều trường (text, title, url).

2. Bật BM25: Cấu hình collection này để nó xây dựng chỉ mục (index) BM25 trên các trường văn bản.

3. Index Dữ liệu: Đẩy final_chunks (từ Phần 2) vào Weaviate.

4. Tìm kiếm: Thực hiện các truy vấn BM25 (text search) trực tiếp trên Weaviate.1.

## Khởi tạo Kết nối

In [18]:
import weaviate
import weaviate.classes.config as wvconfig  # Để định nghĩa schema
import sys
from tqdm import tqdm

# Biến này sẽ giữ các chunks của chúng ta từ Phần 2
if 'final_chunks' not in locals():
    print("Lỗi: Biến `final_chunks` không tồn tại.")
    print("Vui lòng chạy lại code ở Phần 2.4 để tạo chunks.")
    sys.exit("Dừng thực thi. Cần `final_chunks`.")
else:
    print(f"Sẵn sàng index {len(final_chunks)} chunks vào Weaviate.")

# 1. Kết nối với Weaviate
try:
    client = weaviate.connect_to_local(
        host="localhost",
        port=8080,
        grpc_port=50051
    )
    client.is_ready()
    print("✅ Kết nối Weaviate thành công!")
except Exception as e:
    print(f"❌ Lỗi kết nối Weaviate: {e}")
    sys.exit("Dừng thực thi. Kiểm tra Docker Weaviate.")


Sẵn sàng index 110 chunks vào Weaviate.
✅ Kết nối Weaviate thành công!


## Định nghĩa Schema (Collection)

In [ ]:
COLLECTION_NAME = "WikiChunk"

# 1) Xóa collection cũ (nếu có)
if client.collections.exists(COLLECTION_NAME):
    print(f"Đã tìm thấy Collection '{COLLECTION_NAME}' cũ. Đang xóa...")
    client.collections.delete(COLLECTION_NAME)
    print("Đã xóa.")

# 2) Tạo collection mới (BM25 là mặc định cho TEXT; không cần IndexType)
print(f"Đang tạo Collection '{COLLECTION_NAME}' mới...")
wiki_collection = client.collections.create(
    name=COLLECTION_NAME,

    # Thuộc tính (fields)
    properties=[
        wvconfig.Property(
            name="text",
            data_type=wvconfig.DataType.TEXT,
            # BM25 được bật mặc định cho TEXT trong inverted index
        ),
        wvconfig.Property(
            name="title",
            data_type=wvconfig.DataType.TEXT,
        ),
        wvconfig.Property(
            name="url",
            data_type=wvconfig.DataType.TEXT,
            # Nếu không muốn search theo field này, đơn giản là
            # đừng đưa "url" vào danh sách properties khi query.bm25(...)
        ),
    ],

    # Tắt vectorizer vì ta chỉ dùng BM25/text search
    vectorizer_config=wvconfig.Configure.Vectorizer.none(),

    # (Tùy chọn) Nếu muốn chỉnh tham số BM25 ở cấp collection:
    # inverted_index_config=wvconfig.Configure.inverted_index(
    #     bm25_k1=1.2,
    #     bm25_b=0.75
    # ),
)

print(f"✅ Đã tạo Collection '{wiki_collection.name}' thành công.")

Đang tạo Collection 'WikiChunk' mới...


d:\anaconda3\envs\falcon\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


✅ Đã tạo Collection 'WikiChunk' thành công.


## Index Dữ liệu (Đẩy Chunks vào Weaviate)

In [21]:
print(f"Chuẩn bị đẩy {len(final_chunks)} chunks vào Weaviate...")

# 1. Lấy collection object
try:
    wiki_collection = client.collections.get(COLLECTION_NAME)
except weaviate.exceptions.UnexpectedStatusCodeException:
    print(f"Lỗi: Không tìm thấy collection '{COLLECTION_NAME}'.")
    sys.exit("Vui lòng chạy lại bước 3.3.")

# 2. Chuyển đổi dữ liệu sang định dạng Weaviate
# Weaviate cần một list các dictionary, 
# mỗi key là TÊN của property
data_objects = [
    {
        "text": chunk.page_content,
        "title": chunk.metadata.get("title", ""), # Dùng .get() để an toàn
        "url": chunk.metadata.get("url", "")
    }
    for chunk in final_chunks
]

# 3. Sử dụng chế độ batch của Weaviate
print("Đang index dữ liệu (sử dụng batch)...")
with wiki_collection.batch.dynamic() as batch:
    for data in tqdm(data_objects):
        batch.add_object(
            properties=data
            # Chúng ta không cần cung cấp 'uuid'
            # Weaviate sẽ tự tạo
        )

print(f"✅ Index hoàn tất! Đã thêm {len(data_objects)} chunks.")
# Đóng client sau khi xong việc
client.close()

Chuẩn bị đẩy 110 chunks vào Weaviate...
Đang index dữ liệu (sử dụng batch)...


100%|██████████| 110/110 [00:00<00:00, 5875.12it/s]


✅ Index hoàn tất! Đã thêm 110 chunks.


## Text search

In [ ]:
import weaviate
import sys
from weaviate.classes.query import MetadataQuery  # if you use it

# (Optional) if you might have an old client object in memory:
try:
    if "client" in globals() and hasattr(client, "close"):
        client.close()
except Exception:
    pass

client = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)

try:
    # your logic here
    wiki_collection = client.collections.get(COLLECTION_NAME)
    print("✅ Kết nối và lấy collection thành công.")

    query_text = "Hợp đồng thông minh là gì?"
    top_k = 3
    print(f"\n--- {top_k} kết quả cho query: '{query_text}' (chỉ tìm trên 'text') ---")

    response = wiki_collection.query.bm25(
        query=query_text,
        query_properties=["text"],            # for 4.17.0
        limit=top_k,
        return_metadata=MetadataQuery(score=True)
    )

    for i, obj in enumerate(response.objects, 1):
        score = obj.metadata.score
        score_str = f"{score:.4f}" if score is not None else "N/A"
        props = obj.properties or {}
        print(f"\n[Kết quả {i}] (BM25 score: {score_str})")
        print(f"Nguồn: {props.get('title', '(không có)')}")
        print(f"URL: {props.get('url', '(không có)')}")
        print("Nội dung chunk (300 ký tự đầu):")
        print((props.get('text', '')[:300].strip() + '...') if props.get('text') else '(trống)')
        print('-' * 20)

except Exception as e:
    print(f"❌ Lỗi: {e}")
    # avoid bare sys.exit() without closing; we close in finally
finally:
    client.close()   # <-- always close

✅ Kết nối và lấy collection thành công.

--- 3 kết quả cho query: 'Hợp đồng thông minh là gì?' (chỉ tìm trên 'text') ---

[Kết quả 1] (BM25 score: 3.9309)
Nguồn: Ethereum
URL: https://vi.wikipedia.org/wiki/Ethereum
Nội dung chunk (300 ký tự đầu):
Lưu ý rằng "hợp đồng" trong Ethereum không phải là một cái gì đó phải "hoàn thành" hoặc "tuân thủ". Thay vào đó, nó giống như các "thực thể tự trị" sống bên trong môi trường Ethereum, luôn thực hiện một đoạn mã cụ thể khi được tác động bởi một thông điệp hoặc giao dịch, và có quyền kiểm soát trực số...
--------------------

[Kết quả 2] (BM25 score: 3.0850)
Nguồn: Blockchain
URL: https://vi.wikipedia.org/wiki/Blockchain
Nội dung chunk (300 ký tự đầu):
Hợp đồng thông minh (smart contracts) và tài sản thông minh
Hợp đồng thông minh là các khối để xây dựng nên các ứng dụng phi tập trung. Một hợp đồng thông minh tương đương với một chương trình nhỏ mà bạn có thể tin tưởng với một đơn vị giá trị và quản lý giá trị đó. Ý tưởng cơ bản đằng sau hợp đồn

## Nâng cao: Text Search trên Nhiều trường

In [26]:
import weaviate
from weaviate.classes.query import MetadataQuery

# Kết nối lại
client = weaviate.connect_to_local()
try:
    wiki_collection = client.collections.get(COLLECTION_NAME)
    print("✅ Kết nối & lấy collection ok.")

    query_text = "Hợp đồng thông minh Ethereum"
    top_k = 3
    search_fields = ["title", "text"]  # BM25 hỗ trợ list field, KHÔNG hỗ trợ 'title^2'

    print(f"\n--- {top_k} kết quả cho query: '{query_text}' ---")
    print(f"    (Tìm trên {search_fields})")

    response = wiki_collection.query.bm25(
        query=query_text,
        query_properties=search_fields,                # đúng tham số cho 4.17.x
        limit=top_k,
        return_metadata=MetadataQuery(score=True)      # lấy BM25 score
    )

    for i, obj in enumerate(response.objects, 1):
        score = obj.metadata.score
        score_str = f"{score:.4f}" if score is not None else "N/A"
        props = obj.properties or {}
        print(f"\n[Kết quả {i}] (BM25: {score_str})")
        print(f"Nguồn: {props.get('title', '(không có)')}")
        print(f"URL: {props.get('url', '(không có)')}")
        text = (props.get('text') or '').strip()
        print("Nội dung chunk (300 ký tự đầu):")
        print((text[:300] + "...") if text else "(trống)")
        print("-" * 20)
finally:
    client.close()


✅ Kết nối & lấy collection ok.

--- 3 kết quả cho query: 'Hợp đồng thông minh Ethereum' ---
    (Tìm trên ['title', 'text'])

[Kết quả 1] (BM25: 4.5898)
Nguồn: Ethereum
URL: https://vi.wikipedia.org/wiki/Ethereum
Nội dung chunk (300 ký tự đầu):
Các điểm khác biệt cơ bản so với Bitcoin
Về nguồn gốc, Bitcoin được tạo ra như một loại tiền tệ và để lưu trữ giá trị. Còn Ethereum được tạo ra như một nền tảng giao dịch hợp đồng thông minh phân tán. Lưu ý rằng Bitcoin cũng có thể xử lý được hợp đồng thông minh, và Ethereum cũng có thể được sử dụng...
--------------------

[Kết quả 2] (BM25: 4.4678)
Nguồn: Ethereum
URL: https://vi.wikipedia.org/wiki/Ethereum
Nội dung chunk (300 ký tự đầu):
Chúng có thể được sử dụng để tạo điều kiện, xác minh và thực thi việc đàm phán hoặc thực hiện các hướng dẫn thủ tục kinh tế và có khả năng tránh được sự kiểm duyệt, thông đồng và rủi ro từ phía đối tác. Trong Ethereum, các hợp đồng thông minh được coi là các kịch bản tự trị hoặc các ứng dụng phân cấ...
------

In [27]:
# [BÀI TẬP]

# 1. Đặt câu truy vấn
student_query = "Web3 là gì?"
top_k_student = 2

# [CODE CỦA HỌC VIÊN TẠI ĐÂY]
# Gợi ý:
# 1. Kết nối với Weaviate và lấy collection 'WikiChunk'.
# 2. Định nghĩa 'query_properties' để tìm trên cả 'text' và 'title'
#    (lần này không cần dấu ^).
# 3. Gọi hàm 'wiki_collection.query.bm25(...)' với các tham số.
# 4. Lặp qua 'response.objects' và in kết quả.
# 5. Nhớ đóng client.

# (Học viên sẽ code ở đây)

# Phần 5: Index và Search với Elasticsearch (Sử dụng Native Client)

In [28]:
from elasticsearch import Elasticsearch
from elasticsearch import helpers # Để index hàng loạt (bulk)
import sys
from tqdm import tqdm

# Kiểm tra lại `final_chunks` từ Phần 2
if 'final_chunks' not in locals():
    print("Lỗi: Biến `final_chunks` không tồn tại.")
    print("Vui lòng chạy lại code ở Phần 2.4 để tạo chunks.")
    sys.exit("Dừng thực thi. Cần `final_chunks`.")
else:
    print(f"Sẵn sàng index {len(final_chunks)} chunks vào Elasticsearch.")

# 1. Kết nối với Elasticsearch
try:
    es_client = Elasticsearch("http://localhost:9200")
    if es_client.ping():
        print("✅ Kết nối Elasticsearch thành công!")
    else:
        print("❌ Lỗi: Không thể ping Elasticsearch. Kiểm tra Docker.")
        sys.exit("Dừng thực thi.")
except Exception as e:
    print(f"❌ Lỗi khi kết nối Elasticsearch: {e}")
    sys.exit("Dừng thực thi.")

Sẵn sàng index 110 chunks vào Elasticsearch.
✅ Kết nối Elasticsearch thành công!


In [29]:
INDEX_NAME = "wiki_chunk_es"

# 1. Định nghĩa Mapping
# Đây là cấu trúc JSON mô tả dữ liệu của chúng ta
es_mapping = {
    "properties": {
        # 1. Trường nội dung (text search)
        "text": {
            "type": "text" 
            # "type": "text" trong ES ngụ ý sử dụng 
            # chỉ mục inverted index và thuật toán BM25 
            # (hoặc TF-IDF tùy phiên bản/cấu hình)
        },
        # 2. Trường tiêu đề (text search)
        "title": {
            "type": "text"
        },
        # 3. Trường URL (không cần search, chỉ lưu)
        "url": {
            "type": "keyword", # "keyword" là 1 chuỗi không phân tích
            "index": False     # Không xây dựng index cho trường này
        }
    }
}

# 2. Xóa index cũ (nếu có) để làm lại từ đầu
if es_client.indices.exists(index=INDEX_NAME):
    print(f"Đã tìm thấy index '{INDEX_NAME}' cũ. Đang xóa...")
    es_client.indices.delete(index=INDEX_NAME)
    print("Đã xóa.")

# 3. Tạo index mới với mapping
try:
    print(f"Đang tạo index '{INDEX_NAME}' mới...")
    es_client.indices.create(index=INDEX_NAME, mappings=es_mapping)
    print(f"✅ Đã tạo index '{INDEX_NAME}' thành công.")
except Exception as e:
    print(f"❌ Lỗi khi tạo index: {e}")
    sys.exit("Dừng thực thi.")

Đang tạo index 'wiki_chunk_es' mới...
✅ Đã tạo index 'wiki_chunk_es' thành công.


In [30]:
def generate_es_actions(chunks, index_name):
    """
    Hàm generator để tạo "actions" cho bulk API của ES.
    """
    for chunk in chunks:
        yield {
            "_index": index_name, # Index để thêm vào
            "_source": {         # Dữ liệu của chúng ta
                "text": chunk.page_content,
                "title": chunk.metadata.get("title", ""),
                "url": chunk.metadata.get("url", "")
            }
        }

# 1. Tạo các actions
print(f"Chuẩn bị {len(final_chunks)} actions cho bulk index...")
actions = generate_es_actions(final_chunks, INDEX_NAME)

# 2. Thực hiện bulk index
try:
    print("Đang index dữ liệu (sử dụng helpers.bulk)...")
    # tqdm bọc helpers.bulk để xem tiến trình
    success, failed = helpers.bulk(
        es_client, 
        actions, 
        chunk_size=500, # Gửi 500 docs/lần
        request_timeout=60,
        # Bọc 'actions' bằng tqdm nếu muốn xem tiến trình chi tiết
        # (nhưng helpers.bulk đã trả về_
    ) 
    print(f"✅ Index hoàn tất! {success} thành công, {failed} thất bại.")
    
    # Yêu cầu ES refresh index để sẵn sàng tìm kiếm ngay
    es_client.indices.refresh(index=INDEX_NAME)
    print("Đã refresh index, sẵn sàng tìm kiếm.")
    
except Exception as e:
    print(f"❌ Lỗi trong quá trình bulk index: {e}")

Chuẩn bị 110 actions cho bulk index...
Đang index dữ liệu (sử dụng helpers.bulk)...


C:\Users\Admin\AppData\Local\Temp\ipykernel_15272\4205544079.py:23: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  success, failed = helpers.bulk(


✅ Index hoàn tất! 110 thành công, [] thất bại.
Đã refresh index, sẵn sàng tìm kiếm.


In [31]:
# 1. Đặt câu truy vấn
query_text = "Hợp đồng thông minh là gì?"
top_k = 3

# 2. Xây dựng Query DSL
# Chúng ta chỉ tìm trên trường 'text'
es_query_body = {
    "size": top_k, # Giống 'limit'
    "query": {
        "multi_match": {
            "query": query_text,
            "fields": ["text"] # Chỉ tìm trên trường này
        }
    }
}

print(f"\n--- {top_k} kết quả cho query: '{query_text}' (chỉ tìm trên 'text') ---")

# 3. Thực hiện tìm kiếm
response = es_client.search(
    index=INDEX_NAME,
    body=es_query_body
)

# 4. In kết quả
# Kết quả của ES nằm trong 'response['hits']['hits']'
for i, hit in enumerate(response['hits']['hits']):
    score = hit['_score'] # Đây là điểm BM25
    source = hit['_source'] # Đây là dữ liệu gốc
    
    print(f"\n[Kết quả {i+1}] (Score BM25: {score:.2f})")
    print(f"Nguồn: {source['title']}")
    print(f"URL: {source['url']}")
    print("Nội dung chunk (300 ký tự đầu):")
    print(source['text'][:300].strip() + "...")
    print("-" * 20)


--- 3 kết quả cho query: 'Hợp đồng thông minh là gì?' (chỉ tìm trên 'text') ---

[Kết quả 1] (Score BM25: 8.67)
Nguồn: Ethereum
URL: https://vi.wikipedia.org/wiki/Ethereum
Nội dung chunk (300 ký tự đầu):
Lưu ý rằng "hợp đồng" trong Ethereum không phải là một cái gì đó phải "hoàn thành" hoặc "tuân thủ". Thay vào đó, nó giống như các "thực thể tự trị" sống bên trong môi trường Ethereum, luôn thực hiện một đoạn mã cụ thể khi được tác động bởi một thông điệp hoặc giao dịch, và có quyền kiểm soát trực số...
--------------------

[Kết quả 2] (Score BM25: 6.80)
Nguồn: Blockchain
URL: https://vi.wikipedia.org/wiki/Blockchain
Nội dung chunk (300 ký tự đầu):
Hợp đồng thông minh (smart contracts) và tài sản thông minh
Hợp đồng thông minh là các khối để xây dựng nên các ứng dụng phi tập trung. Một hợp đồng thông minh tương đương với một chương trình nhỏ mà bạn có thể tin tưởng với một đơn vị giá trị và quản lý giá trị đó. Ý tưởng cơ bản đằng sau hợp đồng...
--------------------

[Kết quả 3] (Scor

In [32]:
# 1. Đặt câu truy vấn
query_text = "Hợp đồng thông minh Ethereum"
top_k = 3

# 2. Xây dựng Query DSL (với boost)
# Chú ý 'fields': ["title^2", "text"]
es_query_boosted = {
    "size": top_k,
    "query": {
        "multi_match": {
            "query": query_text,
            "fields": [
                "title^2", # Ưu tiên tiêu đề gấp 2 lần
                "text"
            ] 
        }
    }
}

print(f"\n--- {top_k} kết quả cho query: '{query_text}' (boost 'title^2') ---")

# 3. Thực hiện tìm kiếm
response = es_client.search(
    index=INDEX_NAME,
    body=es_query_boosted
)

# 4. In kết quả
for i, hit in enumerate(response['hits']['hits']):
    score = hit['_score']
    source = hit['_source']
    
    print(f"\n[Kết quả {i+1}] (Score BM25: {score:.2f})")
    print(f"Nguồn: {source['title']}")
    print(f"URL: {source['url']}")
    print("Nội dung chunk (300 ký tự đầu):")
    print(source['text'][:300].strip() + "...")
    print("-" * 20)


--- 3 kết quả cho query: 'Hợp đồng thông minh Ethereum' (boost 'title^2') ---

[Kết quả 1] (Score BM25: 8.69)
Nguồn: Ethereum
URL: https://vi.wikipedia.org/wiki/Ethereum
Nội dung chunk (300 ký tự đầu):
Các điểm khác biệt cơ bản so với Bitcoin
Về nguồn gốc, Bitcoin được tạo ra như một loại tiền tệ và để lưu trữ giá trị. Còn Ethereum được tạo ra như một nền tảng giao dịch hợp đồng thông minh phân tán. Lưu ý rằng Bitcoin cũng có thể xử lý được hợp đồng thông minh, và Ethereum cũng có thể được sử dụng...
--------------------

[Kết quả 2] (Score BM25: 8.47)
Nguồn: Ethereum
URL: https://vi.wikipedia.org/wiki/Ethereum
Nội dung chunk (300 ký tự đầu):
Chúng có thể được sử dụng để tạo điều kiện, xác minh và thực thi việc đàm phán hoặc thực hiện các hướng dẫn thủ tục kinh tế và có khả năng tránh được sự kiểm duyệt, thông đồng và rủi ro từ phía đối tác. Trong Ethereum, các hợp đồng thông minh được coi là các kịch bản tự trị hoặc các ứng dụng phân cấ...
--------------------

[Kết quả 3] (Score BM2

In [33]:
# [BÀI TẬP]

# 1. Đặt câu truy vấn
student_query = "Web3 là gì?"
top_k_student = 2

# [CODE CỦA HỌC VIÊN TẠI ĐÂY]
# Gợi ý:
# 1. Xây dựng một dictionary 'es_query_body_student'.
# 2. Đặt 'size' là 'top_k_student'.
# 3. Tạo một 'query' > 'multi_match'.
# 4. Đặt 'query' là 'student_query'.
# 5. Đặt 'fields' là một list gồm "text" và "title" (không có boost).
# 6. Gọi 'es_client.search(...)' với 'body' là dict bạn vừa tạo.
# 7. Lặp qua 'response['hits']['hits']' và in kết quả.

# (Học viên sẽ code ở đây)

# Phần 6: Tìm kiếm Thông tin trên Internet (Web Retrievers)

In [35]:
from duckduckgo_search import DDGS
import json

query = "Ethereum"
top_k = 3

print(f"--- {top_k} tin tức mới nhất về: '{query}' ---")

try:
    with DDGS(timeout=20) as ddgs:
        # Ưu tiên tin tức thay vì text
        results = ddgs.news(
            keywords=query,
            region="vi-vn",         # hoặc "wt-wt"
            safesearch="moderate",
            timelimit="w",          # d=ngày, w=tuần, m=tháng
            max_results=top_k
        )

        if not results:
            # fallback sang web thường, backend 'lite' để tránh bị chặn
            results = ddgs.text(
                keywords=f"Tin tức mới nhất {query}",
                region="vi-vn",
                safesearch="moderate",
                timelimit="w",
                backend="lite",      # 'auto'|'html'|'lite'|'bing' (tài liệu mới)
                max_results=top_k
            )

        if not results:
            print("Không tìm thấy kết quả nào (có thể do mạng bị chặn/ratelimit).")
        else:
            for i, res in enumerate(results):
                print(f"\n[Kết quả {i+1}]")
                # news(): 'title','body','url','date','image','source'
                # text(): 'title','body','href'
                title = res.get("title")
                body  = res.get("body")
                url   = res.get("url") or res.get("href")
                date  = res.get("date")
                src   = res.get("source")

                if date: print(f"  Ngày: {date}")
                if src:  print(f"  Nguồn: {src}")
                print(f"  Tiêu đề: {title}")
                print(f"  Mô tả: {body}")
                print(f"  URL: {url}")

except Exception as e:
    print(f"Lỗi khi tìm kiếm DuckDuckGo: {e}")


--- 3 tin tức mới nhất về: 'Ethereum' ---


C:\Users\Admin\AppData\Local\Temp\ipykernel_15272\300679516.py:10: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS(timeout=20) as ddgs:



[Kết quả 1]
  Ngày: 2025-11-05T00:57:00+00:00
  Nguồn: TheStreet
  Tiêu đề: Crypto analyst warns of 50% drop for Ethereum in 'worst-case scenario'
  Mô tả: On Oct. 5, Bitcoin (BTC) was exchanging hands at $102,499, down 1% in a day. Ethereum (ETH) went further to dip 4.5% as it trades at $3,332.10 at press time. Popular crypto analyst Ali Martinez, known as @ali_charts on X, predicted a worst-case scenario for Ether in which the cryptocurrency could fall as low as $1,700.
  URL: https://www.msn.com/en-us/money/markets/crypto-analyst-predicts-ethereums-worst-case-scenario/ar-AA1POkPU

[Kết quả 2]
  Ngày: 2025-11-08T03:12:00+00:00
  Nguồn: Seeking Alpha
  Tiêu đề: Ethereum: Near-Term Headwinds, And A Long-Term Bullish Case
  Mô tả: Ethereum (ETH-USD) trades 32% below its ATH, offering long-term accumulation potential. Explore current market trends, ETF flows, and network fundamentals.
  URL: https://seekingalpha.com/article/4840769-ethereum-near-term-headwinds-and-a-long-term-bullish-ca

In [38]:
import wikipedia

# Cấu hình ngôn ngữ
wikipedia.set_lang("vi")

# 1. Tìm kiếm
search_term = "Hợp đồng thông minh"
top_k = 5
print(f"--- {top_k} trang Wiki hàng đầu cho: '{search_term}' ---")

results = wikipedia.search(search_term, results=top_k)

if not results:
    print("Không tìm thấy kết quả.")
else:
    for i, title in enumerate(results):
        print(f"[Kết quả {i+1}] {title}")

# 2. Lấy nội dung trang đầu tiên
if results:
    first_title = results[0]
    print(f"\nĐang tải nội dung trang: {first_title}")
    page = wikipedia.page(first_title)
    print(page.summary[:500] + "...")


--- 5 trang Wiki hàng đầu cho: 'Hợp đồng thông minh' ---
[Kết quả 1] Bitcoin
[Kết quả 2] Blockchain
[Kết quả 3] Ethereum
[Kết quả 4] Lăng Chủ tịch Hồ Chí Minh
[Kết quả 5] Diem (tiền mã hóa)

Đang tải nội dung trang: Bitcoin
Bitcoin (ký hiệu: BTC, XBT, ) là một loại tiền mã hóa, được phát minh bởi một cá nhân hoặc tổ chức vô danh dùng tên Satoshi Nakamoto dưới dạng phần mềm mã nguồn mở từ năm 2009. Bitcoin có thể được trao đổi trực tiếp bằng thiết bị kết nối Internet mà không cần thông qua một tổ chức tài chính trung gian nào.
Bitcoin có cách hoạt động khác hẳn so với các loại tiền tệ điển hình: không có một ngân hàng trung ương nào quản lý nó và hệ thống hoạt động dựa trên một giao thức mạng ngang hàng trên Interne...


In [39]:
# [BÀI TẬP]

# 1. Đặt câu truy vấn
student_query = "Các ứng dụng của blockchain trong y tế"
top_k_student = 2

# [CODE CỦA HỌC VIÊN TẠI ĐÂY]
# Gợi ý:
# 1. Sử dụng 'with DDGS() as ddgs:'.
# 2. Gọi hàm 'ddgs.text(...)' với 'keywords' và 'max_results'.
# 3. Lặp qua 'results' và in 'title' và 'body' (snippet).

# (Học viên sẽ code ở đây)